## Fine-tuning Stable Diffusion XL with DreamBooth and LoRA on Naruto Data

SDXL consists of a much larger UNet and two text encoders that make the cross-attention context quite larger than the previous variants. In this notebook we fine-tune this large model using DreamBooth and LoRA.

## Install Dependencies

In [ ]:
# Install dependencies.
!pip install bitsandbytes transformers accelerate peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.3 MB/s eta 0:00:00


In [ ]:
!pip install git+https://github.com/huggingface/diffusers.git -q
!pip install datasets -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py

--2025-11-19 19:52:32--  https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 86775 (85K) [text/plain]
Saving to: ‘train_dreambooth_lora_sdxl.py’

train_dreambooth_lo 100%[===================>]  84.74K  --.-KB/s    in 0.004s  

2025-11-19 19:52:34 (18.8 MB/s) - ‘train_dreambooth_lora_sdxl.py’ saved [86775/86775]



In [ ]:
# Force UTF-8 encoding for colab and run Hugging Face Accelerate default configuration
import locale
locale.getpreferredencoding = lambda: "UTF-8"

!accelerate config default

accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [ ]:
# Huggingface login to upload model weights
from huggingface_hub import notebook_login
notebook_login()

## Dataset
DreamBooth only requires a few images to learn style transfer. Hence, we sample 50 random images from the dataset.

In [1]:
import os
import random
import json
from datasets import load_dataset
from PIL import Image

num_images = 50 # Number of images we want to sample.

ds = load_dataset("lambdalabs/naruto-blip-captions", split="train") # Load the Naruto Dataset

save_dir = "naruto_dreambooth_data" # Directory where we will save the data
os.makedirs(save_dir, exist_ok=True)

indices = random.sample(range(len(ds)), num_images)

print(f"Saving {num_images} random Naruto images and metadata.jsonl...")

# Create metadata JSON file
metadata_path = os.path.join(save_dir, "metadata.jsonl")
with open(metadata_path, "w") as outfile:

    for i, idx in enumerate(indices):
        example = ds[idx]

        # Save image
        img = example["image"].convert("RGB")
        file_name = f"naruto_{i:04d}.jpg"
        img.save(os.path.join(save_dir, file_name))

        # Append "in Naruto animation style" to caption
        caption = example["text"].split("\n")[0]  # take first caption line
        caption = caption + " in Naruto animation style"

        # Add to JSON
        entry = {
            "file_name": file_name,
            "prompt": caption
        }
        json.dump(entry, outfile)
        outfile.write("\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

data/train-00000-of-00002-12944970063701(…):   0%|          | 0.00/344M [00:00<?, ?B/s]

data/train-00001-of-00002-cefa2f480689f1(…):   0%|          | 0.00/357M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Saving 50 random Naruto images and metadata.jsonl...


## Training

In [ ]:
#!/usr/bin/env bash
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --dataset_name="naruto_dreambooth_data" \
  --output_dir="naruto_LoRA" \
  --caption_column="prompt" \
  --mixed_precision="fp16" \
  --instance_prompt="Naruto anime character" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=3 \
  --gradient_checkpointing \
  --learning_rate=1e-4 \
  --snr_gamma=5.0 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --use_8bit_adam \
  --max_train_steps=500 \
  --checkpointing_steps=25 \
  --seed="0"

2025-11-19 20:07:38.745322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763582858.788019    4745 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763582858.802184    4745 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763582858.836595    4745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763582858.836628    4745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763582858.836632    4745 computation_placer.cc:177] computation placer alr

## Save Model to Hub

In [ ]:
from huggingface_hub import whoami
from pathlib import Path
from train_dreambooth_lora_sdxl import save_model_card
from huggingface_hub import upload_folder, create_repo
USERNAME = 'sprodem'
output_dir = "Naruto_Drambooth_LoRA"
repo_id = f"{USERNAME}/{output_dir}"

In [ ]:
repo_id = create_repo(repo_id, exist_ok=True).repo_id

# change the params below according to your training arguments
save_model_card(
    repo_id = repo_id,
    images=[],
    base_model="stabilityai/stable-diffusion-xl-base-1.0",
    train_text_encoder=False,
    instance_prompt="Naruto anime character",
    validation_prompt=None,
    repo_folder=output_dir,
    vae_path="madebyollin/sdxl-vae-fp16-fix",
    use_dora=False
)

upload_folder(
    repo_id=repo_id,
    folder_path="naruto_LoRA",  # Here add the path to the folder where you saved the checkpoints
    commit_message="End of training",
    ignore_patterns=["step_*", "epoch_*"],
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint-250/optimizer.bin:   3%|2         |  388kB / 15.1MB            

  ..._lora_weights.safetensors:   0%|          | 94.1kB / 23.4MB            

  ...eckpoint-25/scheduler.bin: 100%|##########| 1.40kB / 1.40kB            

...

  ..._lora_weights.safetensors:   0%|          | 94.1kB / 23.4MB            

  ..._lora_weights.safetensors:   0%|          | 94.1kB / 23.4MB            

  .../checkpoint-250/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

  ..._lora_weights.safetensors:   0%|          | 94.1kB / 23.4MB            

CommitInfo(commit_url='https://huggingface.co/sprodem/Naruto_Drambooth_LoRA/commit/ee1b4fd1422864233a0da2cf6fdf6bdf99ef05d7', commit_message='End of training', commit_description='', oid='ee1b4fd1422864233a0da2cf6fdf6bdf99ef05d7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sprodem/Naruto_Drambooth_LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='sprodem/Naruto_Drambooth_LoRA'), pr_revision=None, pr_num=None)

In [7]:
from IPython.display import display, Markdown

link_to_model = f"https://huggingface.co/{repo_id}"
display(Markdown("Access model here {}".format(link_to_model)))

Access model here https://huggingface.co/sprodem/Naruto_Drambooth_LoRA